# Module 7b: Auto Loader + incremental silver for Bank B (credit card)
Same pattern as bank A: **files → bronze (Auto Loader) → silver (foreachBatch + MERGE)**.

Differences from bank A: 4 columns, ISO dates (`yyyy-MM-dd`), one amount column + `DR`/`CR` type, no reference number.

Run the cells **one at a time** and check each output.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

base        = "/Volumes/workspace/finance/raw_files"
landing     = f"{base}/landing"
checkpoints = f"{base}/_checkpoints"

BRONZE = "workspace.finance.bronze_bank_b"
SILVER = "workspace.finance.silver_transactions"

## Part A: statements arrive in the landing folder
This time we copy all six months at once (you already saw the "3 now, 3 later" demo with bank A).

In [0]:
for m in ["04", "05", "06", "07", "08", "09"]:
    dbutils.fs.cp(f"{base}/bank_b/bank_b_card_2026_{m}.csv",
                  f"{landing}/bank_b/bank_b_card_2026_{m}.csv")

display(dbutils.fs.ls(f"{landing}/bank_b"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_b/bank_b_card_2026_04.csv,bank_b_card_2026_04.csv,842,1793774284000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_b/bank_b_card_2026_05.csv,bank_b_card_2026_05.csv,1158,1793774284000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_b/bank_b_card_2026_06.csv,bank_b_card_2026_06.csv,938,1793774285000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_b/bank_b_card_2026_07.csv,bank_b_card_2026_07.csv,1067,1793774286000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_b/bank_b_card_2026_08.csv,bank_b_card_2026_08.csv,979,1793774287000
dbfs:/Volumes/workspace/finance/raw_files/landing/bank_b/bank_b_card_2026_09.csv,bank_b_card_2026_09.csv,530,1793774287000


## Part B: bronze with Auto Loader
Its own checkpoint folder (`bronze_bank_b`): every stream needs its own.

In [0]:
schema_b = "date_raw STRING, description_raw STRING, amount_raw STRING, type_raw STRING"

def load_bronze_bank_b():
    before = spark.table(BRONZE).count() if spark.catalog.tableExists(BRONZE) else 0

    stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(schema_b)
        .load(f"{landing}/bank_b")
        .withColumn("source_file", F.col("_metadata.file_name"))
        .withColumn("ingested_at", F.current_timestamp()))

    (stream.writeStream
        .option("checkpointLocation", f"{checkpoints}/bronze_bank_b")
        .trigger(availableNow=True)
        .toTable(BRONZE)
        .awaitTermination())

    after = spark.table(BRONZE).count()
    print(f"bronze: {before} -> {after}  (+{after - before} rows)")

load_bronze_bank_b()

bronze: 0 -> 94  (+94 rows)


In [0]:
# Which files are in bronze?
display(spark.sql(f"""
  SELECT source_file, count(*) AS rows, max(ingested_at) AS ingested_at
  FROM {BRONZE} GROUP BY source_file ORDER BY source_file"""))

source_file,rows,ingested_at
bank_b_card_2026_04.csv,14,2026-11-04T06:38:20.661Z
bank_b_card_2026_05.csv,20,2026-11-04T06:38:20.661Z
bank_b_card_2026_06.csv,16,2026-11-04T06:38:20.661Z
bank_b_card_2026_07.csv,18,2026-11-04T06:38:20.661Z
bank_b_card_2026_08.csv,17,2026-11-04T06:38:20.661Z
bank_b_card_2026_09.csv,9,2026-11-04T06:38:20.661Z


In [0]:
# Run again with no new files: expect +0 rows (the checkpoint remembers)
load_bronze_bank_b()

bronze: 94 -> 94  (+0 rows)


## Part C: incremental silver with `foreachBatch` + `MERGE`

In [0]:
def to_amount(c):
    return F.expr(f"try_cast(regexp_replace(trim({c}), ',', '') AS DECIMAL(12,2))")

def clean_bank_b(df):
    kind = F.upper(F.trim("type_raw"))
    return df.select(
        F.expr("to_date(try_to_timestamp(trim(date_raw), 'yyyy-MM-dd'))").alias("txn_date"),  # ISO dates
        F.trim("description_raw").alias("description"),
        F.lit(None).cast("string").alias("bank_ref"),                                          # no ref number
        F.abs(to_amount("amount_raw")).alias("amount"),                                         # one amount column
        F.when(kind.isin("DR", "DEBIT"), F.lit("DEBIT"))
         .when(kind.isin("CR", "CREDIT"), F.lit("CREDIT")).alias("direction"),                 # DR/CR
        "source_file",
    ).withColumn("account", F.lit("bank_b"))

def add_txn_id(df):
    key = F.concat_ws("|", "account", F.col("txn_date").cast("string"), F.upper("description"),
                      F.col("amount").cast("string"), "direction", F.coalesce("bank_ref", F.lit("")))
    w = Window.partitionBy("source_file", "dedupe_key").orderBy(F.lit(1))
    return (df.withColumn("dedupe_key", key)
              .withColumn("occurrence", F.row_number().over(w))
              .withColumn("txn_id", F.sha2(F.concat_ws("|", "dedupe_key", "occurrence"), 256))
              .drop("dedupe_key", "occurrence"))

def upsert_bank_b(batch_df, batch_id):
    valid = clean_bank_b(batch_df).filter(
        "txn_date IS NOT NULL AND amount > 0 AND direction IS NOT NULL")   # the empty-amount row is skipped

    w = Window.partitionBy("txn_id").orderBy("source_file")
    new_rows = (add_txn_id(valid)
        .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
        .withColumn("loaded_at", F.current_timestamp()))

    new_rows.createOrReplaceTempView("new_rows_b")
    new_rows.sparkSession.sql(f"""
        MERGE INTO {SILVER} AS t
        USING new_rows_b AS s
        ON t.txn_id = s.txn_id
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
def load_silver_bank_b():
    before = spark.table(SILVER).count()
    (spark.readStream.table(BRONZE)
        .writeStream
        .foreachBatch(upsert_bank_b)
        .option("checkpointLocation", f"{checkpoints}/silver_bank_b")
        .trigger(availableNow=True)
        .start()
        .awaitTermination())
    after = spark.table(SILVER).count()
    print(f"silver: {before} -> {after}  (+{after - before} rows)")

# Silver already has these months from Module 4, so expect +0
load_silver_bank_b()

silver: 275 -> 275  (+0 rows)


## Part D: October's card statement arrives

In [0]:
october_card = """Transaction Date,Description,Amount,Type
2026-10-03,NETFLIX.COM SUBSCRIPTION,649.00,DR
2026-10-08,POS 453212******1234 AMAZON PAY IN,"2,149.00",DR
"""
dbutils.fs.put(f"{landing}/bank_b/bank_b_card_2026_10.csv", october_card, overwrite=True)

load_bronze_bank_b()   # expect +2 rows
load_silver_bank_b()   # expect +2 rows

Wrote 147 bytes.
bronze: 94 -> 96  (+2 rows)
silver: 275 -> 277  (+2 rows)


In [0]:
# The new October rows from both banks in silver
display(spark.sql(f"""
  SELECT account, txn_date, description, amount, direction, source_file
  FROM {SILVER} WHERE txn_date >= '2026-10-01' ORDER BY account, txn_date"""))

account,txn_date,description,amount,direction,source_file
bank_a,2026-10-01,NEFT-N9988776655-ACME TECHNOLOGIES PVT LTD-SALARY OCT,95000.00,CREDIT,bank_a_statement_2026_10.csv
bank_a,2026-10-05,IMPS-900000000002-RAJESH KUMAR-HOUSE RENT,18000.00,DEBIT,bank_a_statement_2026_10.csv
bank_a,2026-10-07,UPI/SWIGGY/swiggy@axisbank/Food order/900000003,412.00,DEBIT,bank_a_statement_2026_10.csv
bank_b,2026-10-03,NETFLIX.COM SUBSCRIPTION,649.00,DEBIT,bank_b_card_2026_10.csv
bank_b,2026-10-08,POS 453212******1234 AMAZON PAY IN,2149.00,DEBIT,bank_b_card_2026_10.csv


In [0]:
# Every streaming run in bronze's history
display(spark.sql(f"DESCRIBE HISTORY {BRONZE}")
        .select("version", "timestamp", "operation", "operationMetrics.numOutputRows"))

version,timestamp,operation,numOutputRows
2,2026-11-04T06:38:47.000Z,STREAMING UPDATE,2
1,2026-11-04T06:38:23.000Z,STREAMING UPDATE,94
0,2026-11-04T06:38:18.000Z,CREATE TABLE,null
